In [4]:
from pathlib import Path
import pandas as pd

# ==============================================================
# CONFIGURATION
# ==============================================================

BASE_OUTPUTS_DIR = Path("outputs/")

LANGUAGES = ["acehnese", "javanese", "indonesian", "hitu", "minangkabau", "vietnamese"]
PROMPT_TYPES = ["few_shots_only", "few_shots_and_word_candidates", "topic_related"]

RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_OUTPUTS_DIR exists:", BASE_OUTPUTS_DIR.exists())

def find_per_model_output_files(lang: str, prompt_type: str) -> list:
    outputs_dir = BASE_OUTPUTS_DIR / prompt_type
    pattern = f"{lang}_{prompt_type}_*.csv"
    files = sorted(outputs_dir.glob(pattern))
    files = [f for f in files if not f.name.endswith(".bak")]
    return files


def load_all_rows() -> pd.DataFrame:
    all_dfs = []

    for prompt_type in PROMPT_TYPES:
        for lang in LANGUAGES:
            files = find_per_model_output_files(lang, prompt_type)

            if not files:
                print(f"[{lang}/{prompt_type}] no files found, skipping")
                continue

            for f in files:
                df = pd.read_csv(f)
                df["language"] = lang
                df["prompt_type"] = prompt_type

                if "model" not in df.columns or "is_thinking_on" not in df.columns:
                    stem = f.stem
                    prefix = f"{lang}_{prompt_type}_"
                    suffix = stem[len(prefix):] if stem.startswith(prefix) else stem
                    if "model" not in df.columns:
                        df["model"] = suffix.rsplit("_thinking_", 1)[0]
                    if "is_thinking_on" not in df.columns:
                        df["is_thinking_on"] = suffix.endswith("_on")

                all_dfs.append(df)

    if not all_dfs:
        raise RuntimeError("No output files found -- check BASE_OUTPUTS_DIR / LANGUAGES / PROMPT_TYPES.")

    combined = pd.concat(all_dfs, ignore_index=True)
    return combined


combined = load_all_rows()
print(f"Loaded {len(combined)} total rows")

combined.head()

BASE_OUTPUTS_DIR exists: True
Loaded 54000 total rows


,prompt_id,language,length_category,unit,target_length_desc,prompt_type,topic,prompt,output_id,output,...,full_response,model,is_thinking_on,is_valid,fluency_score,cultural_relevance_score,pct_loan_words,seed,candidate_seed,sentence_match_rate
0,b8641c12-b541-45b1-bde2-740053f3baaa,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,8d21c92e-0625-4b69-9aec-825f4a44c2c5,Geudubng meu kawan that.,...,"{""id"": ""msg_011CecP26ARnddtBSBQ7ar7p"", ""contai...",claude-haiku-4-5,False,True,2.0,0.0,0.25,NaN,NaN,NaN
1,a1b43170-baf5-430e-9d58-287c1b3be16e,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,77ebae05-7d89-4416-a10b-7c27b90a2347,Gaki nyoe makan nasi goreng.,...,"{""id"": ""msg_011CecP26BfYdtaQj13QDope"", ""contai...",claude-haiku-4-5,False,True,1.0,4.0,0.20,NaN,NaN,NaN
2,ae51a88a-5190-4e4e-a30b-42c96c3593c3,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,96f96e61-fbd7-4ce8-8421-ae5444f0a964,Gajah nyoe cukôp besar banget.,...,"{""id"": ""msg_011CecP268gvBTPSatj1xHez"", ""contai...",claude-haiku-4-5,False,True,3.0,0.0,0.40,NaN,NaN,NaN
3,4e57ed68-22e6-426d-b1cc-c2cb38751fc0,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,25965002-c1e9-4873-8a50-b19919acd8cf,Geucok meu babah that nyoe.,...,"{""id"": ""msg_011CecP26GNy1W1sr6gbsWgF"", ""contai...",claude-haiku-4-5,False,True,1.0,0.0,0.00,NaN,NaN,NaN
4,3facf64f-3224-4940-a17a-37c9a41559a4,acehnese,5_words,sentence,5 words,few_shots_only,NaN,NaN,a899a39e-8eba-4bc0-a78e-f14d61c3017d,Rumoh meuë gampong that likôt.,...,"{""id"": ""msg_011CecP26CucoV69AQRWhNf2"", ""contai...",claude-haiku-4-5,False,True,1.0,0.0,0.20,NaN,NaN,NaN


In [5]:
import re
from functools import lru_cache
import pandas as pd

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
DICTIONARIES_DIR = "../dictionaries"
NS = [1, 2, 3, 4, 5]


def tokenize(text: str):
    return re.findall(r"\w+", str(text).lower())


@lru_cache(maxsize=None)
def load_dictionary(language):
    """Load + cache the dictionary word set for a language."""
    dictionary_df = pd.read_csv(f"{DICTIONARIES_DIR}/{language}.csv")
    words = set()
    for word in dictionary_df["word"].dropna().astype(str):
        words |= set(tokenize(word))
    return words


# ------------------------------------------------------------------
# Valid unique n-gram stats (n = 1..5)
# ------------------------------------------------------------------
def ngram_stats(texts, dictionary_words, ns=NS):
    """
    Pool n-grams across many outputs without crossing output boundaries.

    For each n:
      total_ngrams        = ALL n-grams (valid or not)
      unique_valid_ngrams = UNIQUE n-grams whose words are ALL in the dictionary
      distinct_n          = unique_valid_ngrams / total_ngrams
    """
    unique_valid = {n: set() for n in ns}
    total = {n: 0 for n in ns}

    for t in texts:
        tokens = tokenize(t)
        # Precompute validity once per token
        is_valid = [w in dictionary_words for w in tokens]

        for n in ns:
            for i in range(len(tokens) - n + 1):
                total[n] += 1
                if all(is_valid[i:i + n]):
                    unique_valid[n].add(tuple(tokens[i:i + n]))

    stats = {}
    for n in ns:
        n_unique = len(unique_valid[n])
        stats[f"total_{n}grams"] = total[n]
        stats[f"unique_valid_{n}grams"] = n_unique
        stats[f"distinct_{n}"] = n_unique / total[n] if total[n] > 0 else 0
    return stats

In [ ]:
# ==============================================================
# 3. AGGREGATE RESULTS
# ==============================================================
df = combined.copy()
df["is_thinking_on"] = df["is_thinking_on"].astype(bool)

group_cols_1 = ["language", "length_category", "prompt_type"]

group_cols_2 = [
    "language",
    "model",
    "is_thinking_on",
    "length_category",
    "prompt_type"
]

rows = []

# --- Calculate n-gram statistics per model configuration ---
for (language, model, thinking, length_cat, prompt_type), g in df.groupby(group_cols_2):
    dictionary_words = load_dictionary(language)
    outputs = g["output"].dropna().astype(str).tolist()

    stats = ngram_stats(outputs, dictionary_words)
    avg_distinct = sum(stats[f"distinct_{n}"] for n in NS) / len(NS)

    rows.append({
        "language": language,
        "model": model,
        "is_thinking_on": thinking,
        "length_category": length_cat,
        "prompt_type": prompt_type,
        "n_outputs": len(outputs),
        "avg_distinct_1_5": avg_distinct,
        **stats
    })

per_model_df = pd.DataFrame(rows)


# --- Group 1: Language, Length Category, and Prompt Type ---
lexical_summary_1 = (
    per_model_df.groupby(group_cols_1, dropna=False)
    .agg(
        n_outputs=("n_outputs", "sum"),
        lexical_diversity=("avg_distinct_1_5", "mean")
    )
    .reset_index()
)

print("\n=== SUMMARY 1: LANGUAGE, LENGTH & PROMPT TYPE ===")
display(lexical_summary_1)

lexical_summary_1.to_csv(
    RESULTS_DIR / "overall_lexical_diversity.csv",
    index=False,
    encoding="utf-8"
)


# --- Group 2: Language, Model, Thinking, Length, and Prompt Type ---
lexical_summary_2 = (
    per_model_df.groupby(group_cols_2, dropna=False)
    .agg(
        n_outputs=("n_outputs", "sum"),
        lexical_diversity=("avg_distinct_1_5", "mean"),
    )
    .reset_index()
)

print("\n=== SUMMARY 2: LANGUAGE, MODEL, THINKING, LENGTH & PROMPT TYPE ===")
display(lexical_summary_2)

lexical_summary_2.to_csv(
    RESULTS_DIR / "detailed_lexical_diversity.csv",
    index=False,
    encoding="utf-8"
)

print(f"\nSaved detailed and summary CSVs to {RESULTS_DIR}/")


=== SUMMARY 1: LANGUAGE, LENGTH & PROMPT TYPE ===


,language,length_category,prompt_type,n_outputs,lexical_diversity
0,acehnese,10_words,few_shots_and_word_candidates,600,0.660078
1,acehnese,10_words,few_shots_only,600,0.230201
2,acehnese,10_words,topic_related,600,0.189142
3,acehnese,15_words,few_shots_and_word_candidates,600,0.643574
4,acehnese,15_words,few_shots_only,600,0.213302
...,...,...,...,...,...
85,vietnamese,5_words,few_shots_only,600,0.456535
86,vietnamese,5_words,topic_related,600,0.459121
87,vietnamese,paragraph,few_shots_and_word_candidates,600,0.711908
88,vietnamese,paragraph,few_shots_only,600,0.447990



=== SUMMARY 2: LANGUAGE, MODEL, THINKING, LENGTH & PROMPT TYPE ===


,language,model,is_thinking_on,length_category,prompt_type,n_outputs,lexical_diversity,distinct_1,distinct_2,distinct_3,distinct_4,distinct_5
0,acehnese,claude-haiku-4-5,False,10_words,few_shots_and_word_candidates,100,0.719996,0.289133,0.685493,0.849315,0.890469,0.885572
1,acehnese,claude-haiku-4-5,False,10_words,few_shots_only,100,0.143440,0.141509,0.231850,0.179045,0.107034,0.057762
2,acehnese,claude-haiku-4-5,False,10_words,topic_related,100,0.097599,0.120087,0.175245,0.110335,0.055195,0.027132
3,acehnese,claude-haiku-4-5,False,15_words,few_shots_and_word_candidates,100,0.724072,0.284477,0.705837,0.852321,0.889401,0.888325
4,acehnese,claude-haiku-4-5,False,15_words,few_shots_only,100,0.142446,0.130467,0.232186,0.171453,0.109628,0.068493
...,...,...,...,...,...,...,...,...,...,...,...,...
535,vietnamese,gpt-5.4-mini,True,5_words,few_shots_only,100,0.446637,0.177645,0.341646,0.468439,0.582090,0.663366
536,vietnamese,gpt-5.4-mini,True,5_words,topic_related,100,0.553467,0.303089,0.483254,0.616352,0.669725,0.694915
537,vietnamese,gpt-5.4-mini,True,paragraph,few_shots_and_word_candidates,100,0.736606,0.115027,0.683205,0.922160,0.975793,0.986847
538,vietnamese,gpt-5.4-mini,True,paragraph,few_shots_only,100,0.452058,0.075161,0.318567,0.504286,0.635048,0.727226



Saved detailed and summary CSVs to results/
